In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

storage_account = "youssef1235445"
container_name = "bronze"

data_location = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"

df_bronze = (
    spark.read.format("parquet")
    .load(data_location)
)

display(df_bronze)

In [0]:
df_bronze_check = (
    spark.read
    .format("parquet")
    .load(data_location)
)

# display(
#     df_bronze_check.filter(F.col("Date_ID") == "DT01248")
# )

display(df_bronze_check.limit(10)) 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Remove duplicates
df_silver = df_bronze.dropDuplicates()

# Casting columns
df_silver = (
    df_silver
    .withColumn("Day", F.col("Day").cast("int"))
    .withColumn("Month", F.col("Month").cast("int"))
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("Units_Sold", F.col("Units_Sold").cast("int"))
    .withColumn("Revenue", F.col("Revenue").cast("double"))
)

# Get string columns
string_cols = [
    field.name for field in df_silver.schema.fields
    if isinstance(field.dataType, StringType)
]

# Fill missing string values
df_silver = df_silver.fillna("Unknown", subset=string_cols)

# Trim spaces
for col_name in string_cols:
    df_silver = df_silver.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

# Revenue business logic
df_silver = df_silver.withColumn(
    "Revenue",
    F.when(
        (F.col("Units_Sold") == 0) & (F.col("Revenue").isNull()),
        0
    ).otherwise(F.col("Revenue"))
)

# Calculate mean revenue for valid rows
mean_revenue = (
    df_silver
    .filter(
        (F.col("Revenue").isNotNull()) &
        (F.col("Revenue") > 0)
    )
    .agg(F.avg("Revenue"))
    .collect()[0][0]
)


df_silver = df_silver.withColumn(
    "Revenue",
    F.when(
        (F.col("Units_Sold") > 0) & (F.col("Revenue").isNull()),
        mean_revenue
    ).otherwise(F.col("Revenue"))
)

# Remove invalid negative values
df_silver = df_silver.filter(
    (F.col("Units_Sold") >= 0) &
    (F.col("Revenue") >= 0)
)

#  Outlier filtering 
quantiles = df_silver.approxQuantile("Revenue", [0.25, 0.75], 0.05)
Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR  

df_silver = df_silver.filter(F.col("Revenue") <= upper_bound)

display(df_silver.limit(10))

In [0]:
storage_account = "youssef1235445"
container_name_silver = "silver"
output_path = f"abfss://{container_name_silver}@{storage_account}.dfs.core.windows.net/sales_data_silver"


(df_silver.write
  .format("delta")               
  .mode("append")             
  .save(output_path)             
)

print("Data successfully saved to Silver layer in ADLS Gen2!")

In [0]:
df_check = (
    spark.read
    .format("delta")
    .load(output_path)
)

# display(
#     df_check.filter(F.col("Date_ID") == "DT01248")
# )

display(df_check.limit(10))